# 15 — Build the environments

Each op runs in its own environment. They are built on the first call that
needs one, so you can skip this notebook and everything still works — the
first call just takes several minutes instead of seconds.

Reasons to run it anyway:

- Build before the session, not during it.
- See what is happening as the environment builds
- In the future, if you update scikit-ops, you can rerun this type of code to see if any of the environments need rebuilt

In [1]:
from skop.runner import Runner

runner = Runner()

ENVIRONMENTS = [
    'minimal',       # toy ops
    'cellpose3',     # Cellpose 3
    'pytorch',       # Cellpose 4, micro-sam
    'stardist-tf',   # StarDist
]

## What do we have?

- `missing` — never built.
- `stale` — built from a different definition than the one installed now.
- `up to date` — nothing to do.


In [2]:
for env in ENVIRONMENTS:
    print(f'{env:14} {runner.environment_status(env)}')

minimal        up to date
cellpose3      up to date
pytorch        up to date
stardist-tf    up to date


## Build

`ensure_environment` builds only what is missing or stale

First time building can be minutes. 

Below we build each environment.  

### minimal — toy ops

In [3]:
runner.ensure_environment('minimal')

minimal: up to date


<appose.builder.BaseBuilder._create_env.<locals>.BuiltEnvironment at 0x71b677618440>

### cellpose3 — Cellpose 3

In [4]:
runner.ensure_environment('cellpose3')

cellpose3: up to date


<appose.builder.BaseBuilder._create_env.<locals>.BuiltEnvironment at 0x71b6776198e0>

### pytorch — Cellpose 4, micro-sam

In [5]:
runner.ensure_environment('pytorch')

pytorch: up to date


<appose.builder.BaseBuilder._create_env.<locals>.BuiltEnvironment at 0x71b677619520>

### stardist-tf — StarDist

In [6]:
runner.ensure_environment('stardist-tf')

stardist-tf: up to date


<appose.builder.BaseBuilder._create_env.<locals>.BuiltEnvironment at 0x71b67761a510>

## Check again

Any you built should now say `up to date`. Any you skipped still say `missing`.

In [7]:
for env in ENVIRONMENTS:
    print(f'{env:14} {runner.environment_status(env)}')

minimal        up to date
cellpose3      up to date
pytorch        up to date
stardist-tf    up to date


## Is the GPU visible?

In the below cell we query the stardist environment that appose and the scikit-ops framework created to see if we have a GPU.   The trick is we use pixi to run the python snippet that looks for the GPU, and point pixi to the location of the scikit-ops stardist-tf pixi.toml.

In [8]:
# Through `pixi run`: calling the env's python directly skips activation and
# reports if there are GPUs in the environment
manifest = runner.env_dir('stardist-tf') / 'pixi.toml'

#!pixi run --manifest-path {manifest} python -c "import tensorflow as tf; print('GPUs:', tf.config.list_physical_devices('GPU'))" 2>&1 | grep -Ei 'gpus:|could not load'
!pixi run --manifest-path {manifest} python -c "import tensorflow as tf; print(\"GPUs:\", tf.config.list_physical_devices(\"GPU\"))" 

2026-09-27 15:37:16.117846: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-27 15:37:16.144664: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-27 15:37:16.144692: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-27 15:37:16.145461: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-27 15:37:16.149696: I tensorflow/core/platform/cpu_feature_guar